## Setup & Data Fetching

!pip install ultralytics roboflow
import torch
import cv2
import os
import glob
import shutil
from ultralytics import YOLO
from roboflow import Roboflow

# Fetch Dataset
rf = Roboflow(api_key="YOUR_API_KEY_HERE")
project = rf.workspace("jeets-workspace-nr9d6").project("kirana-kosh")
clean_data = project.version(3).download("yolov8")
aug_data = project.version(4).download("yolov8")

## Unit I: Data Augmentation as Inductive Bias

# Initialize a blank YOLOv8-Nano model
model_clean = YOLO('yolov8n.pt')

# Train on the clean dataset
results_clean = model_clean.train(
    data=f"{clean_data.location}/data.yaml",
    epochs=25,
    imgsz=640,
    batch=16,
    project="Kirana_Unit1",
    name="clean_model"
)

In [ ]:
# Initialize a fresh YOLOv8-Nano model
model_aug = YOLO('yolov8n.pt')

# Train on the augmented dataset
results_aug = model_aug.train(
    data=f"{aug_data.location}/data.yaml",
    epochs=25,
    imgsz=640,
    batch=16,
    project="Kirana_Unit1",
    name="augmented_model"
)

## Unit II: Feature Pyramid Surgery

import cv2
import os
import glob
import shutil
from ultralytics import YOLO

# 1. Define base paths
base_dir = clean_data.location
test_corrupt_dir = os.path.join(base_dir, "test_corrupt")
img_out_dir = os.path.join(test_corrupt_dir, "images")
lbl_out_dir = os.path.join(test_corrupt_dir, "labels")

# Create fresh directories
os.makedirs(img_out_dir, exist_ok=True)

# 2. Corrupt and save images
print("Corrupting test images with heavy Gaussian Blur...")
for img_path in glob.glob(os.path.join(base_dir, "test", "images", "*.jpg")):
    img = cv2.imread(img_path)
    # Apply a strong 15x15 blur
    blurred_img = cv2.GaussianBlur(img, (15, 15), 0)
    base_name = os.path.basename(img_path)
    cv2.imwrite(os.path.join(img_out_dir, base_name), blurred_img)

# 3. Copy the labels over so YOLO can find them
if os.path.exists(lbl_out_dir):
    shutil.rmtree(lbl_out_dir)
shutil.copytree(os.path.join(base_dir, "test", "labels"), lbl_out_dir)

print(f"Successfully linked labels to {len(glob.glob(img_out_dir + '/*.jpg'))} corrupted images.")

# 4. Create the new YAML
yaml_content = f"""
path: {base_dir}
train: train/images
val: test_corrupt/images  # YOLO will naturally look in test_corrupt/labels now
nc: 22
names: ['Bath_and_Body', 'Beverages', 'Biscuits_and_Cookies', 'Chips_and_Wafers', 'Chocolates_and_Sweets', 'Cooking_Oil', 'Dairy_and_Ghee', 'Deodorants_and_Perfumes', 'Dry_Fruits_and_Nuts', 'Grains_Pulses_Flour', 'Hair_Care', 'Health_Drinks', 'Health_and_Baby_Care', 'Health_and_Pharma', 'Household_and_Cleaning', 'Instant_Food', 'Namkeen_and_Snacks', 'Oral_Care', 'Other_Packaged_Goods', 'Sauces_and_Spreads', 'Spices_and_Masala', 'Tea_and_Coffee']
"""
with open("corrupted_data.yaml", "w") as f:
    f.write(yaml_content)

# 5. Evaluate BOTH models on the properly labeled blurred data
model_clean = YOLO('/content/runs/detect/Kirana_Unit1/clean_model/weights/best.pt')
model_aug = YOLO('/content/runs/detect/Kirana_Unit1/augmented_model/weights/best.pt')

print("\n--- Evaluating CLEAN Model on Corrupted Data ---")
metrics_clean_corrupt = model_clean.val(data="corrupted_data.yaml")

print("\n--- Evaluating AUGMENTED Model on Corrupted Data ---")
metrics_aug_corrupt = model_aug.val(data="corrupted_data.yaml")

In [ ]:
import torch
from ultralytics import YOLO

def ablate_and_evaluate(scale_idx, scale_name):
    print(f"\n" + "="*50)
    print(f"SURGERY: Ablating {scale_name} Object Head (Scale P{scale_idx+3})")
    print("="*50)

    # 1. Load a fresh copy of your best trained augmented model
    model = YOLO('/content/runs/detect/Kirana_Unit1/augmented_model/weights/best.pt')

    # 2. Isolate the Detect head (the very last module in YOLOv8)
    detect_head = model.model.model[-1]

    # 3. Perform the ablation (Zero out the weights)
    # YOLOv8's Detect head has two main branches per scale:
    # cv2 (bounding box regression) and cv3 (class probabilities)
    for param in detect_head.cv2[scale_idx].parameters():
        torch.nn.init.zeros_(param)
    for param in detect_head.cv3[scale_idx].parameters():
        torch.nn.init.zeros_(param)

    # 4. Evaluate the ablated model on the CLEAN test set
    # We use the clean data because we want to isolate architectural failures, not corruption failures
    metrics = model.val(data="/content/Kirana-Kosh-3/data.yaml", plots=False)

    return metrics

# Run the three ablation experiments sequentially
metrics_small = ablate_and_evaluate(0, "Small")
metrics_medium = ablate_and_evaluate(1, "Medium")
metrics_large = ablate_and_evaluate(2, "Large")